In [ ]:
# Cell A - Load test prediction files

from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

BASE_DIR = Path("/Users/tonytony/Final Project")
CLEAN_DIR = BASE_DIR / "Data" / "Cleaned"

model_1_test_path = CLEAN_DIR / "gdp_xgboost_model_1_test_predictions.csv"
model_2_test_path = CLEAN_DIR / "gdp_xgboost_model_2_test_predictions.csv"
model_3_test_path = CLEAN_DIR / "gdp_xgboost_model_3_test_predictions.csv"

print("Model 1 path exists:", model_1_test_path.exists())
print("Model 2 path exists:", model_2_test_path.exists())
print("Model 3 path exists:", model_3_test_path.exists())

model_1_test_df = pd.read_csv(model_1_test_path)
model_2_test_df = pd.read_csv(model_2_test_path)
model_3_test_df = pd.read_csv(model_3_test_path)

In [1]:
# Cell B - Keep only the common test sample across all 3 models

join_cols = ["country_code", "feature_year", "target_year"]

m1 = model_1_test_df[
    join_cols + ["country_name", "wb_region", "actual_gdp_next_year", "predicted_gdp_next_year"]
].copy().rename(
    columns={"predicted_gdp_next_year": "pred_model_1"}
)

m2 = model_2_test_df[
    join_cols + ["actual_gdp_next_year", "predicted_gdp_next_year"]
].copy().rename(
    columns={"predicted_gdp_next_year": "pred_model_2"}
)

m3 = model_3_test_df[
    join_cols + ["actual_gdp_next_year", "predicted_gdp_next_year"]
].copy().rename(
    columns={"predicted_gdp_next_year": "pred_model_3"}
)

common_test_df = (
    m1.merge(
        m2[join_cols + ["pred_model_2"]],
        on=join_cols,
        how="inner"
    )
    .merge(
        m3[join_cols + ["pred_model_3"]],
        on=join_cols,
        how="inner"
    )
)

print("Common test rows:", common_test_df.shape[0])
print("Countries in common sample:", common_test_df["country_code"].nunique())
print("Target year range:", int(common_test_df["target_year"].min()), "-", int(common_test_df["target_year"].max()))

display(common_test_df.head())

NameError: name 'model_1_test_df' is not defined

In [ ]:
# Cell C - Build yearly mean actual vs predicted table on common test sample

yearly_compare_df = (
    common_test_df.groupby("target_year", as_index=False)
    .agg(
        n_obs=("actual_gdp_next_year", "size"),
        actual_gdp=("actual_gdp_next_year", "mean"),
        model_1_pred=("pred_model_1", "mean"),
        model_2_pred=("pred_model_2", "mean"),
        model_3_pred=("pred_model_3", "mean"),
    )
    .sort_values("target_year")
    .reset_index(drop=True)
)

display(yearly_compare_df)

In [ ]:
# Cell D - Line chart: Actual vs Predicted on test set

plt.figure(figsize=(12, 6))

plt.plot(
    yearly_compare_df["target_year"],
    yearly_compare_df["actual_gdp"],
    marker="o",
    linewidth=2.8,
    color="#1f4e79",
    label="Actual GDP"
)

plt.plot(
    yearly_compare_df["target_year"],
    yearly_compare_df["model_1_pred"],
    marker="o",
    linewidth=2.2,
    linestyle="--",
    color="#f28e2b",
    label="Model 1 - Baseline"
)

plt.plot(
    yearly_compare_df["target_year"],
    yearly_compare_df["model_2_pred"],
    marker="o",
    linewidth=2.2,
    linestyle="--",
    color="#59a14f",
    label="Model 2 - Extended"
)

plt.plot(
    yearly_compare_df["target_year"],
    yearly_compare_df["model_3_pred"],
    marker="o",
    linewidth=2.2,
    linestyle="--",
    color="#e15759",
    label="Model 3 - Full"
)

plt.title("Actual vs Predicted GDP on the Common Test Set", fontsize=14, fontweight="bold")
plt.xlabel("Target Year")
plt.ylabel("Mean GDP per Capita (US$)")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Cell E - Optional: save yearly line-chart source table

save_path = CLEAN_DIR / "gdp_xgboost_models_1_2_3_test_linechart_common_sample.csv"
yearly_compare_df.to_csv(save_path, index=False)
print("Saved to:", save_path)